# Insight 177 replication

This notebook runs the PyTorch replication for all six markets. It produces both the paper-comparable 2015-2025 out-of-sample results and an expanded-history sensitivity run.

The OOS protocol follows Insight 177: for each Tuesday evaluation date, the model is trained only on the preceding 104 weekly observations, excludes the current Managed Money observation, uses price inputs through the current Tuesday, and scores the prediction only after the realized Managed Money value is available.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from paper_replication import ReplicationConfig, run_replication, MARKETS

DATA_DIR = PROJECT_ROOT / 'data_required'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)
print(PROJECT_ROOT)

## Configuration

`futures_only` is the default Managed Money basis. Change it to `combined` only for a sensitivity run. The first rolling fit uses 1,024 full-batch epochs; later windows warm-start from the previous fit and use 36 epochs.

In [ ]:
config = ReplicationConfig(data_dir=DATA_DIR, basis='futures_only', train_weeks=104, seed=7)
config

## Paper-comparable run

This run retains the two-year warm-up and evaluates forecasts from 2015 through 2025.

In [ ]:
paper_run = run_replication(config, expanded=False)
paper_metrics = paper_run['metrics']
paper_metrics

In [ ]:
paper_predictions = paper_run['predictions']
paper_predictions.to_csv(OUTPUT_DIR / 'model_predictions_paper_window.csv', index=False)
paper_run['weekly_panel'].to_parquet(PROJECT_ROOT / 'derived' / 'weekly_features_paper_window.parquet', index=False)
paper_metrics.to_csv(OUTPUT_DIR / 'metrics_paper_window.csv')
paper_predictions.head()

## Expanded-history run

This run uses all available weekly dates after the same 104-week warm-up. Its metrics should not be compared directly with the published tables unless the date range is restricted.

In [ ]:
expanded_run = run_replication(config, expanded=True)
expanded_metrics = expanded_run['metrics']
expanded_metrics

In [ ]:
expanded_predictions = expanded_run['predictions']
expanded_predictions.to_csv(OUTPUT_DIR / 'model_predictions_expanded.csv', index=False)
expanded_run['weekly_panel'].to_parquet(PROJECT_ROOT / 'derived' / 'weekly_features_expanded.parquet', index=False)
expanded_metrics.to_csv(OUTPUT_DIR / 'metrics_expanded.csv')
expanded_predictions.tail()

## Diagnostics

The plot compares actual and model-predicted normalized Managed Money positions by market. The paper's published figures use position changes for scoring; this plot is a visual diagnostic for level comovement.

In [ ]:
fig, axes = plt.subplots(len(MARKETS), 1, figsize=(14, 18), sharex=True)
for axis, market in zip(axes, MARKETS):
    subset = paper_predictions.loc[paper_predictions['market'].eq(market)]
    axis.plot(subset['report_week_tuesday'], subset['y'], label='Actual normalized position', linewidth=1)
    axis.plot(subset['report_week_tuesday'], subset['predicted_y'], label='Predicted normalized position', linewidth=1)
    axis.set_title(market)
    axis.grid(alpha=0.25)
    axis.legend(loc='upper left')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures_model_vs_actual_levels.png', dpi=160)
plt.show()

## Checks to record

Compare the paper-window metrics with Insight 177's reported cumulative performance. Differences can arise from the source data vintage, futures roll reconstruction, futures-only versus combined COT basis, missing-settlement handling, and exact implementation details of the regularized bias. Keep the expanded-history results separate from the paper-comparable results.